# Cross-run analyses of the hierarchy-design runs

Analyses that cut across the runs of `hierarchy_design_runs.ipynb` (one run per section there) live here, so that
notebook keeps its one-section-per-run structure. The setup cell loads the first-layer trajectory (every 10 epochs) of
every run from `results/designed_init/` through a light loader that reads only the weight stacks; add a run to `RUN_INDEX`
when a new one appears. Sections:

1. Where deviating probes go: lineage in the ODT tree (moved here from section 22 of the runs notebook on 2026-10-10).


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2
from layer_lenses.designed_init import HIERARCHY_TARGETS, NODE_LEVEL, node_lineage, lineage_chance, probe_deviation_lineage
from layer_lenses.init_sweep import load_tree

RESULTS = Path("../results/designed_init")
tree = load_tree(RESULTS / "hierarchy_replayable" / "tree.npz")            # the same COB-ODT (tree seed 5178) in every run

# run number -> (folder, tag or None for a replay folder, first-layer width); the replay folders hold weights.npz written by save_designed_run
RUN_INDEX = {1: ("hierarchy_kaiming_cosine", "linear_free_kaiming_cosine_seed0", 80), 2: ("hierarchy_kaiming_cosine", "linear_frozen_kaiming_cosine_seed0", 80), 3: ("hierarchy_kaiming_cosine_big", "linear_frozen_kaiming_cosine_big_seed0", 62),
             4: ("hierarchy_kaiming_cosine_isolate", "linear_frozen_kaiming_cosine_small_62_seed0", 62), 5: ("hierarchy_kaiming_cosine_isolate", "linear_frozen_kaiming_cosine_big_80_seed0", 80), 6: ("hierarchy_kaiming_cosine_balanced", "linear_frozen_kaiming_cosine_bal_62_seed0", 62),
             7: ("hierarchy_kaiming_cosine_balanced", "linear_frozen_kaiming_cosine_bal_80_seed0", 80), 8: ("hierarchy_kaiming_cosine_pathsign", "linear_frozen_kaiming_cosine_pathsign_80_seed0", 80), 9: ("hierarchy_kaiming_cosine_cueeq", "linear_frozen_kaiming_cosine_cueeq_80_seed0", 80),
             10: ("hierarchy_kaiming_cosine_balpaths", "linear_frozen_kaiming_cosine_balpaths_80_seed0", 80), 11: ("hierarchy_replayable", None, 80), 12: ("hierarchy_replayable_probe001", None, 80), 13: ("hierarchy_branch750_sgd_frozenL1", None, 80),
             14: ("hierarchy_branch750_adam_frozenL1", None, 80), 15: ("hierarchy_branch750_adam_all", None, 80), 16: ("hierarchy_rotlock099", None, 80), 17: ("hierarchy_scalaradam", None, 80), 18: ("hierarchy_scalaradam_lr003", None, 80),
             19: ("hierarchy_scalaradam_lr01", None, 80), 20: ("hierarchy_scalaradam_lr03", None, 80), 21: ("hierarchy_replayable_62", None, 62), 22: ("hierarchy_idealL1", None, 62), 23: ("hierarchy_idealL1_zerocue", None, 62),
             24: ("hierarchy_replayable_probepos", None, 80), 25: ("hierarchy_replayable_w2neg", None, 80)}

def load_weight_stacks(n, keys=("hidden_layers.0.weight",)):
    """Weight stacks (epochs x shape) of run n for the given parameter names, plus the checkpoint epochs; None if the files are absent."""
    folder, tag, _ = RUN_INDEX[n]
    f = RESULTS / folder / ("runs/" + tag + ".npz" if tag else "replay/weights.npz")
    if not f.exists():
        return None
    z = np.load(f); return {"epochs": z["epochs"], "W": {k: z["w__" + k] for k in keys}}

RUNS = {n: r for n in RUN_INDEX if (r := load_weight_stacks(n)) is not None}
print("runs with a stored first-layer trajectory:", sorted(RUNS), "| missing:", sorted(set(RUN_INDEX) - set(RUNS)))

## 1. Where deviating probes go: lineage in the ODT tree, across runs 1-21

A probe that ends captured (|cos| >= 0.95) on a node other than its designed target is a *deviation*. `probe_deviation_lineage`
classifies the captured node relative to the target in the tree (heap indexing: children of node i are 2i+1 and 2i+2, the
split at i sends w_i . x > 0 to the right child 2i+2): `ancestor`, `descendant` or `unrelated`, with the level gap and the
*side*: for a descendant, whether it sits in the target's positive-side subtree; for an ancestor, on which side of the
ancestor the target lies. `chance` is the fraction of the other 30 nodes on the target's lineage (1 for the root, 1/2 for
level 1, 8/30 for level 2, 5/30 for level 3).

**Findings (end state of every run as of run 21; the cell below recomputes them for every run loaded above, run 22 included).**
- Runs 10-21, i.e. every run with the balanced +-10-quanta third-layer paths: 66 deviations, 35 to an ancestor and 31 to a
  descendant, **0 unrelated** (chance level 27%). The descendants are always a direct child (level gap 1); the ancestors are
  the parent in 33 cases and the root in 2 (run 12). The same holds at every checkpoint along the trajectories: no probe of
  runs 10-21 ever captured an unrelated node.
- Runs 1-9: 79 deviations, 61% lineage-related (chance 29%): 31 unrelated, 9 of them in run 2 and 7 in run 4. So the
  observation is a strong tendency under the earlier designs and an exact rule under the current one.
- A second regularity: a probe that climbs to an ancestor always takes the ancestor's pole on the side that contains its
  target (45 of 45 across all runs): -u_2's probe becomes +u_0 (node 2 is the positive child of the root), +u_1's becomes
  -u_0, +u_4's becomes +u_1, -u_5's becomes -u_2. The neuron ends as a coarser gate for the same region of input space.
- Descendants mostly land in the subtree on the probe's own pole side (25 of 31 in runs 10-21, 20 of 38 in runs 1-9): +u_7's
  probe goes to u_16 (the positive child), -u_14's to u_29 (the negative child), -u_7's to u_15, -u_9's to u_19.
- Who deviates in runs 10-21: rows 5 (11 runs), 14 (11), 29 (11), 8 (8), 2 (6), 17 (3); the rest once or twice. These are
  the same substitutes seen run after run, so the deviation is a property of the init and the data, not of the optimizer
  (SGD, AdamW, ScalarAdam, frozen or locked first layer all agree).


In [ ]:
ALL_RUNS = RUNS
rows = []
for n, rr in ALL_RUNS.items():
    t = probe_deviation_lineage(rr["W"]["hidden_layers.0.weight"][-1], tree); t.insert(0, "run", n); rows.append(t)
E = pd.concat(rows, ignore_index=True); dev = E[E.deviated].copy()
print(f"runs loaded: {sorted(ALL_RUNS)}; probes {len(E)}, deviations {len(dev)}, not captured {int((~E.captured).sum())}")
for lab, sel in (("all", dev.run > 0), ("runs 1-9", dev.run <= 9), ("runs 10-21", dev.run >= 10)):
    d = dev[sel]
    if len(d) == 0: continue
    a = d[d.relation == "ancestor"]; s = d[d.relation == "descendant"]
    print(f"{lab}: {len(d)} deviations: ancestor {len(a)}, descendant {len(s)}, unrelated {int((d.relation == 'unrelated').sum())} -> lineage-related {d.relation.isin(['ancestor', 'descendant']).mean() * 100:.0f}% (chance {d.chance.mean() * 100:.0f}%)"
          f"; ancestor pole == side containing the target: {int((np.sign(a.end_cos) == a.side.map({'+': 1, '-': -1})).sum())}/{len(a)}; descendant on the probe's own pole side: {int((s.pole == s.side.map({'+': 1, '-': -1})).sum())}/{len(s)}")
print("unrelated deviations per run:", dev.groupby("run").relation.apply(lambda r: int((r == "unrelated").sum())).to_dict())
display(dev[dev.run >= 10][["run", "probe", "target", "target_level", "pole", "end_node", "end_cos", "relation", "level_gap", "side"]].round(3).reset_index(drop=True))
# along the trajectories: every off-target capture at any checkpoint
traj = []
for n, rr in ALL_RUNS.items():
    W_ = rr["W"]["hidden_layers.0.weight"][:, :30]; C_ = np.abs(W_ / np.linalg.norm(W_, axis=2, keepdims=True) @ tree.w_list.T) >= 0.95
    for p_ in range(30):
        for j in np.nonzero(C_[:, p_].any(0))[0]:
            if j != p_ // 2: traj.append({"run": n, "probe": p_, "target": p_ // 2, "node": int(j), "relation": node_lineage(p_ // 2, int(j))["relation"], "first_epoch": int(rr["epochs"][np.argmax(C_[:, p_, j])])})
T = pd.DataFrame(traj); print("off-target captures at any checkpoint:", T.relation.value_counts().to_dict(), "| unrelated ones in runs:", sorted(T[T.relation == "unrelated"].run.unique().tolist()))